In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/icml_face_data.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/fer2013.tar.gz
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/example_submission.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/test.csv


## Data/Libraries import

In [4]:

import torch # Main PyTorch Library
from torch import nn # Used for creating the layers and loss function
from torch.optim import Adam # Adam Optimizer
import torchvision.transforms as transforms # Transform function used to modify and preprocess all the images
from torch.utils.data import Dataset, DataLoader # Dataset class and DataLoader for creating the objects
from sklearn.preprocessing import LabelEncoder # Label Encoder to encode the classes from strings to numbers
import matplotlib.pyplot as plt # Used for visualizing the images and plotting the training progress
from PIL import Image # Used to read the images from the directory
import pandas as pd # Used to read/create dataframes (csv) and process tabular data
import numpy as np # preprocessing and numerical/mathematical operations
import os # Used to read the images path from the directory
import wandb
device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU, change cuda to mps if you have a mac
print("Device available: ", device)
torch.manual_seed(42)

Device available:  cuda


In [5]:
from sklearn.model_selection import train_test_split

df = pd.read_csv("/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv")
df.groupby('emotion').size()

# Disgust is inbalanced so we do stratified split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42,stratify=df['emotion'])

In [45]:
class CustomDataset(Dataset):
    def __init__(self, df, transform=None):
        # Parse all pixel strings once, upfront
        pixels = np.stack([
            np.array(p.split(), dtype=np.float32).reshape(1, 48, 48) / 255.0
            for p in df['pixels'].values
        ])
        self.pixels = torch.tensor(pixels)  # shape: (N, 1, 48, 48)
        self.labels = df['emotion'].values
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.pixels[idx]
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

In [46]:
train_dataset = CustomDataset(train_df,transform=None)
val_dataset = CustomDataset(val_df,transform=None)

In [8]:
BATCH_SIZE = 64
LR = 0.001

In [47]:
train_loader = DataLoader(train_dataset, batch_size=4*BATCH_SIZE, shuffle=True,
                           num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=4*BATCH_SIZE, shuffle=True,
                         num_workers=4, pin_memory=True, persistent_workers=True)

In [10]:
import wandb
!wandb login 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






wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


## Helper Functions

In [29]:
import copy

def sanity_check(model, train_loader, model_name, LR, device, optimizer, criterion, n_classes=7, overfit_batch_size=4):

    run = wandb.init(
        entity="ldavi22-free-university-of-tbilisi-",
        project="Emotion Recognition",
        name=f"Sanity check {model_name}",
        config={
            "learning_rate": LR,
            "architecture": "Custom CNN Architecture",
            "dataset": "fer2013",
            "epochs": 10,
        },
    )

    # Work on copies so the real model/optimizer aren't mutated by this check
    model_copy = copy.deepcopy(model).to(device)
    optimizer_copy = type(optimizer)(model_copy.parameters(), lr=LR)

    images, labels = next(iter(train_loader))
    images, labels = images.to(device), labels.to(device)

    # --- Loss @ init check ---
    model_copy.eval()

    with torch.no_grad():
        output = model_copy(images)
        loss_init = criterion(output, labels)
        probs = torch.softmax(output, dim=1)

        expected_loss = -torch.log(torch.tensor(1.0 / n_classes)).item()

        wandb.log({
            "init_check/loss": loss_init.item(),
            "init_check/expected_loss": expected_loss,
            "init_check/sample_prob": probs[0][0].item(),
            "init_check/prob_sum": probs[0].sum().item(),
        })

        print(f"Loss @ init: {loss_init.item():.4f} | Expected (-log(1/{n_classes})): {expected_loss:.4f}")

    # --- Overfit a tiny batch ---
    model_copy.train()

    tiny_images = images[:overfit_batch_size]
    tiny_labels = labels[:overfit_batch_size]

    for step in range(200):
        output = model_copy(tiny_images)
        loss = criterion(output, tiny_labels)

        optimizer_copy.zero_grad()
        loss.backward()
        optimizer_copy.step()

        acc = (output.argmax(dim=1) == tiny_labels).float().mean()

        wandb.log({
            "overfit/loss": loss.item(),
            "overfit/acc": acc.item(),
            "step": step
        })

    wandb.finish()

In [41]:
def train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=None, epochs=50, run_name="run",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra=None, checkpoint_path="best_model.pt",
                 augment_fn=None, device=None):

    config = {"epochs": epochs}
    if config_extra:
        config.update(config_extra)

    run = wandb.init(entity=entity, project=project, name=run_name, config=config)
    wandb.watch(model, criterion, log="all", log_freq=10)

    n_train_samples = len(train_loader.dataset)
    n_val_samples = len(val_loader.dataset)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        total_loss_train = 0
        total_acc_train = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            if augment_fn is not None:
                inputs = augment_fn(inputs)

            output = model(inputs)
            loss = criterion(output, labels)
            total_loss_train += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            total_acc_train += (torch.argmax(output, axis=1) == labels).sum().item()

        model.eval()
        total_loss_val = 0
        total_acc_val = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                output = model(inputs)
                loss = criterion(output, labels)
                total_loss_val += loss.item()
                total_acc_val += (torch.argmax(output, axis=1) == labels).sum().item()

        avg_train_loss = total_loss_train / len(train_loader)
        avg_val_loss = total_loss_val / len(val_loader)
        avg_train_acc = (total_acc_train / n_train_samples) * 100
        avg_val_acc = (total_acc_val / n_val_samples) * 100

        history["train_loss"].append(avg_train_loss)
        history["train_acc"].append(avg_train_acc)
        history["val_loss"].append(avg_val_loss)
        history["val_acc"].append(avg_val_acc)

        if scheduler is not None:
            scheduler.step(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), checkpoint_path)

        wandb.log({
            "train/loss": avg_train_loss,
            "train/acc": avg_train_acc,
            "val/loss": avg_val_loss,
            "val/acc": avg_val_acc,
            "lr": optimizer.param_groups[0]["lr"],
            "epoch": epoch
        })

    artifact = wandb.Artifact(name=f"model-{run.name.replace(' ', '-')}", type="model")
    artifact.add_file(checkpoint_path)
    wandb.log_artifact(artifact)

    wandb.finish()
    return history

## Model Definition


In [37]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1a = nn.Conv2d(1,64,3,padding=1)
        self.bn1a = nn.BatchNorm2d(64)
        self.conv1b = nn.Conv2d(64,64,3,padding=1)
        self.bn1b = nn.BatchNorm2d(64)

        self.conv2a = nn.Conv2d(64,128,3,padding=1)
        self.bn2a = nn.BatchNorm2d(128)
        self.conv2b = nn.Conv2d(128,128,3,padding=1)
        self.bn2b = nn.BatchNorm2d(128)

        self.conv3a = nn.Conv2d(128,256,3,padding=1)
        self.bn3a = nn.BatchNorm2d(256)
        self.conv3b = nn.Conv2d(256,256,3,padding=1)
        self.bn3b = nn.BatchNorm2d(256)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2,2)
        self.dropout2d = nn.Dropout2d(0.2)
        self.dropout = nn.Dropout(0.5)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(256*6*6,256)
        self.output = nn.Linear(256,7)

    def forward(self,x):
        
        x = self.relu(self.bn1a(self.conv1a(x)))
        x = self.relu(self.bn1b(self.conv1b(x)))
        x = self.pool(x)            # 24x24
        x = self.dropout2d(x)

        x = self.relu(self.bn2a(self.conv2a(x)))
        x = self.relu(self.bn2b(self.conv2b(x)))
        x = self.pool(x)            # 12x12
        x = self.dropout2d(x)

        x = self.relu(self.bn3a(self.conv3a(x)))
        x = self.relu(self.bn3b(self.conv3b(x)))
        x = self.pool(x)            # 6x6
        x = self.dropout2d(x)

        x = self.flatten(x)         # 256*6*6 = 9216
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.output(x)
        return x

model = Net().to(device)

In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5,min_lr=1e-5
)

In [15]:
sanity_check(model, train_loader, "VGGStyleNet", LR, device, optimizer, criterion)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: ldavi22 (ldavi22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loss @ init: 1.9494 | Expected (-log(1/7)): 1.9459


init_check/expected_loss,▁
init_check/loss,▁
init_check/prob_sum,▁
init_check/sample_prob,▁
overfit/acc,▁█████▁█████████████████████████████████
overfit/loss,██▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
step,▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇█████
init_check/expected_loss,1.94591
init_check/loss,1.9494
init_check/prob_sum,1
init_check/sample_prob,0.13651


In [16]:
result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=scheduler, epochs=100,
                 run_name="VGGStyleNet + Scheduler ",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": LR,
            "architecture": "VGGStyleNet (3 blocks, 64-128-256)",
            "dataset": "fer2013",
            "epochs": 100,
            "scheduler": "ReduceLROnPlateau",
            "augmentation": "None",
            "batch_size": BATCH_SIZE,
        }, checkpoint_path="vgg_style_v1.pt",
        device=device)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: ldavi22 (ldavi22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


KeyboardInterrupt: 

#### Adding Augmentation

In [21]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
])

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5,min_lr=1e-5
)

result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=scheduler, epochs=100,
                 run_name="VGGStyleNet + Scheduler + Augmentation ",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": LR,
            "architecture": "VGGStyleNet (3 blocks, 64-128-256)",
            "dataset": "fer2013",
            "epochs": 100,
            "scheduler": "ReduceLROnPlateau",
            "augmentation": "flip+rotation+translate",
            "batch_size": BATCH_SIZE,
        }, checkpoint_path="vgg_style_v1.pt",
        device=device,augment_fn = train_transform)

epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇█████
lr,███████████████▄▄▄▄▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▂▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇███████████████
train/loss,█▇▆▆▅▅▅▅▄▄▄▄▄▄▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▃▂▂▃▃▄▃▄▅▅▆▇▆▆▆▇▇▇▇▇▇▇████▇███████████
val/loss,█▅▄▄▄▃▃▂▃▂▂▂▁▂▁▁▁▂▂▁▁▂▂▂▂▂▁▂▂▂▂▂▂▂▂▂▂▂▂▂
epoch,99
lr,1e-05
train/acc,75.47351
train/loss,0.67316
val/acc,67.64194


In [34]:
sanity_check(model, train_loader, "Sanity Check VGGStyleNet + Scheduler + Augmentation", LR, device, optimizer, criterion)


Loss @ init: 1.9470 | Expected (-log(1/7)): 1.9459


init_check/expected_loss,▁
init_check/loss,▁
init_check/prob_sum,▁
init_check/sample_prob,▁
overfit/acc,▁███████████████████████████████████████
overfit/loss,▄█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
step,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▇▇▇▇▇██
init_check/expected_loss,1.94591
init_check/loss,1.94699
init_check/prob_sum,1
init_check/sample_prob,0.15309


In [36]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
])

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.7, patience=7,min_lr=1e-5
)

result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=scheduler, epochs=150,
                 run_name="VGGStyleNet + Scheduler + Augmentation v2",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": LR,
            "architecture": "VGGStyleNet (3 blocks, 64-128-256)",
            "dataset": "fer2013",
            "epochs": 100,
            "scheduler": "ReduceLROnPlateau f=0.7 p=7",
            "augmentation": "flip+rotation+translate",
            "batch_size": BATCH_SIZE,
        }, checkpoint_path="vgg_style_v1.pt",
        device=device,augment_fn = train_transform)

epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇████
lr,███████████████████████▆▆▆▆▆▆▄▄▄▄▃▂▂▂▁▁▁
train/acc,▁▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇███████
train/loss,█▇▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val/acc,▁▁▂▃▄▄▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████████████
val/loss,█▇▆▅▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂
epoch,149
lr,8e-05
train/acc,80.0061
train/loss,0.54593
val/acc,68.32114


#### Tuning Learning Rate

In [38]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3*LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.7, patience=7,min_lr=1e-5
)

In [39]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
])

result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=scheduler, epochs=150,
                 run_name="VGGStyleNet + Scheduler + Augmentation v3",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": 3*LR,
            "architecture": "VGGStyleNet (3 blocks, 64-128-256)",
            "dataset": "fer2013",
            "epochs": 150,
            "scheduler": "ReduceLROnPlateau f=0.7 p=7",
            "augmentation": "flip+rotation+translate",
            "batch_size": BATCH_SIZE,
        }, checkpoint_path="vgg_style_v1.pt",
        device=device,augment_fn = train_transform)

KeyboardInterrupt: 

#### Adding extra augmentation

In [53]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1a = nn.Conv2d(1,64,3,padding=1)
        self.bn1a = nn.BatchNorm2d(64)
        self.conv1b = nn.Conv2d(64,64,3,padding=1)
        self.bn1b = nn.BatchNorm2d(64)

        self.conv2a = nn.Conv2d(64,128,3,padding=1)
        self.bn2a = nn.BatchNorm2d(128)
        self.conv2b = nn.Conv2d(128,128,3,padding=1)
        self.bn2b = nn.BatchNorm2d(128)

        self.conv3a = nn.Conv2d(128,256,3,padding=1)
        self.bn3a = nn.BatchNorm2d(256)
        self.conv3b = nn.Conv2d(256,256,3,padding=1)
        self.bn3b = nn.BatchNorm2d(256)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2,2)
        self.dropout2d = nn.Dropout2d(0.2)
        self.dropout = nn.Dropout(0.5)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(256*6*6,256)
        self.output = nn.Linear(256,7)

    def forward(self,x):
        
        x = self.relu(self.bn1a(self.conv1a(x)))
        x = self.relu(self.bn1b(self.conv1b(x)))
        x = self.pool(x)            # 24x24
        x = self.dropout2d(x)

        x = self.relu(self.bn2a(self.conv2a(x)))
        x = self.relu(self.bn2b(self.conv2b(x)))
        x = self.pool(x)            # 12x12
        x = self.dropout2d(x)

        x = self.relu(self.bn3a(self.conv3a(x)))
        x = self.relu(self.bn3b(self.conv3b(x)))
        x = self.pool(x)            # 6x6
        x = self.dropout2d(x)

        x = self.flatten(x)         # 256*6*6 = 9216
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.output(x)
        return x

model = Net().to(device)

In [55]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.7, patience=7,min_lr=1e-5
)

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
])

result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=scheduler, epochs=150,
                 run_name="VGGStyleNet + Scheduler + Augmentation v4",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": LR,
            "architecture": "VGGStyleNet (3 blocks, 64-128-256)",
            "dataset": "fer2013",
            "epochs": 150,
            "scheduler": "ReduceLROnPlateau f=0.7 p=7",
            "augmentation": "flip+rotation+translate+colorJitter",
            "batch_size": 4*BATCH_SIZE,
        }, checkpoint_path="vgg_style_v1.pt",
        device=device,augment_fn = train_transform)